# Week 5 — hyperparameter search for the covariate HMM

**Purpose.** One job only: search `STATES`, `AR_LAGS` and the full covariate design (calendar *and* weather) for the
autoregressive + covariate-transition HMM on `Room 009` at 30-minute aggregation, and report the
trade-off front. Everything in `week_5.ipynb` that reports a *fitted* model — diagnostics, decoded
states, AIC/BIC tables, forecast plots, the second-order model — lives there, not here.

**Data.** The pre-built split in `{DATA_PATH}/room_009/30min`. Only `y_train` and `y_val` are read;
the test split is never touched by the search. The two are *pooled* into one series and re-cut into
cross-validation folds (below), so the saved train/val boundary is not itself used as a cut. The
saved `y` is already baseline-calibrated (min = 400 ppm), so no calibration happens here.

**Covariates are rebuilt per trial** rather than read from the saved `X`, which is frozen at week
harmonic k=1, time-of-day k=1..3 and the full weather block. The 30-min timestamps are reconstructed
from `metadata.csv`; rebuilding the saved design that way reproduces `X_train.csv` to ~1e-13, which
is what makes the rebuilt columns row-aligned with the saved `y`. Every part of the design is
searched: the off-day flag, the two harmonic counts, and which weather channels are included
(one on/off switch per channel).

**Scoring: 4-fold expanding-window cross-validation.** A single held-out block is one draw of the
occupancy calendar, so a design can win by suiting that particular fortnight rather than by
generalising. Instead each trial is scored on `N_FOLDS = 4` expanding-window folds over the pooled
train+val series (the `TimeSeriesSplit` layout: fold *i* trains on everything up to cut *i* and
validates on the next block). Within every fold the Gaussian+cov model is fitted, the
autoregressive+cov model is warm-started from it, and the AR model is scored by rolling 6-hour-ahead
forecasts over that fold's validation block. **Three objectives, each averaged over the folds**, all held-out:

| | what it asks | reading |
|---|---|---|
| **RMSE** | how far off, in ppm | lower is better; the only one in the units of the data |
| **MASE** | how it compares to a naive random walk over the same 6 h | **< 1 beats** `ŷ_{t+K} = y_t`; > 1 loses to it |
| **MDA** | how often the direction of change is right | share of anchors; 0.5 is a coin flip |

MASE replaces R2 and MDA replaces the correlation. The reason is that both are already comparable
across folds: MASE is a ratio against a baseline evaluated on the *same* window, so a quiet
fortnight scales numerator and denominator alike, and MDA is a proportion. Neither is measured
against a per-fold variance, so both average directly over the four folds — where R2's four
incomparable denominators had to be pooled through a variance-stabilising transform first. Each
fold's coefficient of determination is still recorded in `r2_folds` as a diagnostic, since it says
whether the forecast beats that block's own flat mean, but it is not optimised.

Only the *hyperparameters* are chosen this way; no fitted parameter arrays are averaged across
folds. The selected design is refit in `week_5.ipynb`, which is where the reported model lives.


## Imports and config

In [1]:
import jax
jax.config.update("jax_enable_x64", True)   # must come before anything touches JAX

from pathlib import Path

import numpy as np
import pandas as pd
import jax.numpy as jnp
import optuna

from src.api.v4 import (
    HMM, GaussEmission, AutoregressiveGaussEmission, DynamicTransition, ForwardAlgorithm,
)
from src.base.utils import transition_matrix_to_logits
from drivers.utils import load_train_data, load_val_data, load_base_data_path


/Users/madshaakonsson/Desktop/7-semester/hmm-modelling/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# --- Search target ------------------------------------------------------
DATA_NAME = "room_009"      # {DATA_PATH}/room_009/30min: y_train / y_val / metadata
DATA_TAG  = "30min"
FIT_TOL   = 1e-2            # per-fit LL tolerance, as in the week-5 fits
N_TRIALS  = 60              # a trial is now N_FOLDS fit pairs, so budget ~4x the old search
N_FOLDS   = 4               # expanding-window CV folds over the pooled train+val series

HORIZON_HOURS = 6
BIN_SECONDS   = 1800
VAL_K = int(round(HORIZON_HOURS * 3600 / BIN_SECONDS))   # 12 steps = 6 h ahead

# Candidate weather channels. Nothing here is forced into the design: `objective`
# samples one boolean per column, alongside the off-day flag and the two harmonic
# counts, so the whole covariate design is searched.
WEATHER_COLS = [
    "mean_temp",
    "mean_relative_hum",
    "mean_wind_speed",
    "mean_pressure",
    "mean_cloud_cover",
    "mean_radiation",
]


## Covariate builder

`build_covariates(datetimes, use_off_day, tod_harmonics, week_harmonics, weather)` returns
`(X, names)`: an `(N, D)` matrix of an optional off-day flag, cyclical week / time-of-day Fourier
terms, and the requested subset of hourly weather channels matched by time. Harmonic `k` contributes
`sin/cos(2*pi*k*t / period)`; `weather` is a sequence of names drawn from `WEATHER_COLS`, so an
empty sequence drops the weather block entirely.

In [3]:
# --- Holidays -----------------------------------------------------------
try:
    import holidays as _holidays
    _dk_holidays = _holidays.Denmark()

    def _is_holiday(d):
        return d.date() in _dk_holidays
except ImportError:
    print("Warning: `holidays` package not found, using hard-coded 2024 DK holidays")
    _DK_HOLIDAYS = {
        pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-28"), pd.Timestamp("2024-03-29"),
        pd.Timestamp("2024-03-31"), pd.Timestamp("2024-04-01"), pd.Timestamp("2024-04-26"),
        pd.Timestamp("2024-05-09"), pd.Timestamp("2024-05-19"), pd.Timestamp("2024-05-20"),
        pd.Timestamp("2024-12-25"), pd.Timestamp("2024-12-26"),
    }

    def _is_holiday(d):
        return d.normalize() in _DK_HOLIDAYS

# --- Weather ------------------------------------------------------------

weather_df = pd.read_csv("data/raw/dtu/weather.csv", parse_dates=["DateFrom", "DateTo"])
weather_df = (
    weather_df[["DateFrom", *WEATHER_COLS]].dropna().sort_values("DateFrom").reset_index(drop=True)
)


def _weather_features(datetimes, cols):
    """Match the requested hourly weather channels (UTC) to each local, tz-naive time."""
    cols = list(cols)
    if not cols:
        return np.empty((len(datetimes), 0))
    local = pd.DatetimeIndex(datetimes).tz_localize(
        "Europe/Copenhagen", ambiguous="NaT", nonexistent="shift_forward"
    )
    left = pd.DataFrame({"t": local.tz_convert("UTC")})
    left["order"] = np.arange(len(left))
    left = left.sort_values("t")
    merged = pd.merge_asof(
        left, weather_df.rename(columns={"DateFrom": "t"}), on="t", direction="nearest",
    )
    merged = merged.sort_values("order")
    return merged[cols].ffill().bfill().to_numpy()


# --- Covariate matrix ---------------------------------------------------

def _fourier(angle, harmonics, label):
    """sin/cos pairs for each requested harmonic of a 2*pi-normalised angle."""
    cols, names = [], []
    for k in harmonics:
        cols += [np.sin(k * angle), np.cos(k * angle)]
        names += [f"sin_{label}_{k}", f"cos_{label}_{k}"]
    return cols, names


def build_covariates(datetimes, use_off_day, tod_harmonics, week_harmonics, weather=WEATHER_COLS):
    """Return (X, names): an (N, D) covariate matrix and its D column names.

    The whole design is passed in per trial: an off-day flag, `week_harmonics` /
    `tod_harmonics` sin-cos pairs, and `weather` -- the subset of `WEATHER_COLS`
    to include, which may be empty. With every argument off the matrix has D = 0.
    """
    weather = list(weather)
    dt = pd.DatetimeIndex(datetimes)
    cols, names = [], []

    if use_off_day:
        is_weekend = dt.dayofweek >= 5
        is_holiday = np.array([_is_holiday(d) for d in dt])
        cols.append((is_weekend | is_holiday).astype(float))
        names.append("off_day")

    seconds_into_week = dt.dayofweek * 86400 + dt.hour * 3600 + dt.minute * 60 + dt.second
    c, n = _fourier(2 * np.pi * seconds_into_week / (7 * 86400), week_harmonics, "week")
    cols += c
    names += n

    seconds_into_day = dt.hour * 3600 + dt.minute * 60 + dt.second
    c, n = _fourier(2 * np.pi * seconds_into_day / 86400, tod_harmonics, "tod")
    cols += c
    names += n

    X = np.column_stack(cols) if cols else np.empty((len(dt), 0))
    return np.column_stack([X, _weather_features(datetimes, weather)]), [*names, *weather]

### Small helpers shared with `week_5.ipynb`

In [4]:
def standardise(X, train_mask):
    mean = X[train_mask].mean(axis=0)
    std  = jnp.where(X[train_mask].std(axis=0) == 0, 1.0, X[train_mask].std(axis=0))
    return (X - mean) / std


def state_means(emission, ys):
    """Per-state base means, whichever emission is asked.

    `GaussEmission` exposes them directly as `mu`; the autoregressive emission splits
    them into `mu_vals` (state base) and `mu` (base + AR term on the previous value).
    """
    return getattr(emission, "mu_vals", emission.mu)(0, ys)



## Validation metrics and the multi-lag forecast

`rolling_forecast_ar` is the plug-in multi-step forecast, general in the number of AR lags: it
carries a window of the last `k` values (column `j` holds `y_{t-j}`, the ordering
`AutoregressiveGaussEmission.mu` expects after its flip) and pushes each prediction onto the front
of that window. The `week_5.ipynb` version reads `phi()[0]` only, so it would silently ignore lags
2+ — and this search samples `AR_LAGS` up to 6.


In [5]:
def rmse(y_true, y_pred):
    return float(jnp.sqrt(jnp.mean((y_true - y_pred) ** 2)))


def mase(y_true, y_pred, y_naive):
    """Mean absolute error relative to a naive forecast's, on the same anchors.

    MASE < 1 means the model beats the baseline; MASE > 1 means it does not. The
    baseline passed in is the random walk *at this forecast's own horizon* --
    y_hat_{t+K} = y_t, the last value observed at the anchor -- so the comparison is
    like-for-like: both forecasts stand at the same anchor and aim K steps ahead.

    Note this is not the Hyndman-Koehler scaling, which divides by the in-sample
    *one-step* naive MAE. At K = 12 that denominator is much smaller than the error
    any 6-hour forecast can achieve, so every model would score far above 1 and the
    "< 1 beats naive" reading would be lost. `mase_insample` below reports that
    version too, for comparability with the literature.
    """
    return float(jnp.mean(jnp.abs(y_true - y_pred)) / jnp.mean(jnp.abs(y_true - y_naive)))


def mase_insample(y_true, y_pred, ys_train):
    """Hyndman-Koehler MASE: scaled by the in-sample one-step naive MAE.

    The standard definition, recorded as a diagnostic. Expect values well above 1
    here: the denominator is a one-step error and the numerator a 12-step one.
    """
    return float(jnp.mean(jnp.abs(y_true - y_pred)) / jnp.mean(jnp.abs(jnp.diff(ys_train))))


def mda(y_true, y_pred, y_anchor):
    """Share of anchors where the forecast gets the direction of change right.

    Direction is taken from the anchor: sign(y_{t+K} - y_t) against
    sign(y_hat_{t+K} - y_t) -- did CO2 rise or fall over the horizon, and did the
    model say so. 0.5 is what a coin flip gets. Exact ties (no change either way)
    count as agreement, which is the conventional treatment and vanishingly rare on
    continuous ppm values.
    """
    return float(jnp.mean(jnp.sign(y_true - y_anchor) == jnp.sign(y_pred - y_anchor)))


def r2_score(y_true, y_pred):
    """Out-of-sample coefficient of determination -- a diagnostic, not an objective.

    Recorded per fold because it says whether the forecast beats that block's own
    flat mean, which is worth knowing. It is not optimised: it measures against each
    fold's own SST, and those differ by a factor of ~3 across these windows, so the
    values are not comparable between folds. MASE takes its place as the
    scale-free accuracy measure.
    """
    ss_res = jnp.sum((y_true - y_pred) ** 2)
    ss_tot = jnp.sum((y_true - jnp.mean(y_true)) ** 2)
    return float(1 - ss_res / ss_tot)


def mape(y_true, y_pred):
    return float(jnp.mean(jnp.abs((y_true - y_pred) / y_true)) * 100)


In [6]:
def rolling_forecast_ar(model, ys, xs_std, n_train, K):
    """K-step-ahead plug-in forecast anchored at every held-out observation, any AR order.

    Returns (y_true, y_pred, y_anchor): the realised value K steps after each anchor,
    the forecast of it, and the value at the anchor itself.

    Same structure as `rolling_discrete(..., is_ar=True)` but general in the number
    of lags: `lags[:, j]` is y_{t-j}, matching the flipped slice in
    `AutoregressiveGaussEmission.mu`, and each prediction is pushed onto the front
    of that window as the next step's lag-1 value.
    """
    ys = jnp.asarray(ys)
    T = len(ys)
    out = ForwardAlgorithm().run(model.params, model.u_pre, ys=ys, ts=None, xs=xs_std)
    utt = out.utt.reshape(T, -1)                                              # (T, S)
    # `ts` is ignored by a discrete transition, so unit waiting times are fine.
    Gammas = model.transition.transition_matrices(jnp.arange(T), jnp.ones(T), ys, xs_std)

    base = model.emission.mu_vals(0, ys, xs_std)                              # (S,) state base means
    phi = model.emission.phi()                                                # (k, S)
    k = phi.shape[0]

    anchors = jnp.arange(n_train, T - K)
    u = utt[anchors]
    lags = jnp.stack([ys[anchors - j] for j in range(k)], axis=1)              # (A, k), col 0 = newest

    # mu_s = base_s + sum_j phi_js (y_{t-j} - base_s)
    #      = base_s (1 - sum_j phi_js) + sum_j phi_js y_{t-j}
    intercept = base[None, :] * (1.0 - phi.sum(axis=0))[None, :]              # (1, S)

    y_pred = None
    for m in range(1, K + 1):
        u = jnp.einsum("ai,aij->aj", u, Gammas[anchors + m])
        mu_state = intercept + jnp.einsum("aj,js->as", lags, phi)             # (A, S)
        y_pred = jnp.sum(u * mu_state, axis=1)                                # state-weighted mean
        lags = jnp.concatenate([y_pred[:, None], lags[:, :-1]], axis=1)       # plug-in next lag

    # The anchor values come back too: they are the naive random-walk forecast at
    # this horizon (y_hat_{t+K} = y_t) and the reference point for the direction of
    # change, so MASE and MDA both need them and neither should re-derive `anchors`.
    return np.asarray(ys[anchors + K]), np.asarray(y_pred), np.asarray(ys[anchors])


## The objective

In [7]:
# =====================================================================
# Optuna search over STATES / AR_LAGS / covariate design for the
# Autoregressive + covariate HMM, scored on held-out 6h-ahead forecasts
# averaged over N_FOLDS expanding-window time-series folds.
#
# Data: the pre-built split in {DATA_PATH}/room_009/30min. y_train and y_val
# are pooled into one series and re-cut into folds -- the test split is never
# touched by the search.
# Covariates are *rebuilt* per design (the saved X is frozen at week k=1,
# tod k=1..3 and the full weather block) and z-scored *inside* each fold, on
# that fold's training rows only. The design itself is searched: the off-day
# flag, both harmonic counts, and one inclusion switch per weather channel.
#
# =====================================================================

_SPLIT_CACHE = {}
_DESIGN_CACHE = {}


def load_split(data_name=DATA_NAME, tag=DATA_TAG):
    """(ys_train, ys_val, dates_tv, n_train) -- read once, then cached.

    `dates_tv` is the 30-min timestamp of every train+val row, rebuilt from
    metadata.csv. The split rows are one gap-free segment, so
    `date_range(start, periods=n_bins)` reproduces the recorded start /
    train_end / val_end exactly -- which is what keeps the rebuilt covariates
    row-aligned with the saved y. `n_train` is the saved boundary; the folds
    below ignore it and re-cut the pooled series themselves.
    """
    key = (data_name, tag)
    if key not in _SPLIT_CACHE:
        ys_train, _ = load_train_data(data_name, tag)
        ys_val, _   = load_val_data(data_name, tag)
        meta = pd.read_csv(
            Path(load_base_data_path()) / data_name / tag / "metadata.csv"
        ).iloc[0]
        n_train, n_val = int(meta["n_train"]), int(meta["n_val"])
        dates = pd.date_range(meta["start"], periods=int(meta["n_bins"]), freq=meta["bin"])
        _SPLIT_CACHE[key] = (ys_train, ys_val, dates[: n_train + n_val], n_train)
    return _SPLIT_CACHE[key]


def time_series_folds(n, n_folds=N_FOLDS):
    """[(train_end, val_end), ...]: expanding window, equal-length validation blocks.

    With L = n // (n_folds + 1), fold i trains on [0, (i+1)L) and validates on
    [(i+1)L, (i+2)L) -- the cuts of sklearn's TimeSeriesSplit(n_folds), which is
    why nothing from sklearn is imported for it. The last fold's training block is
    n_folds/(n_folds+1) of the series, i.e. close to the saved train/val boundary;
    the first trains on a fifth of it, which is the shortest window any design has
    to cope with.
    """
    L = n // (n_folds + 1)
    return [((i + 1) * L, (i + 2) * L) for i in range(n_folds)]


def count_free_params(model, n_frozen=1):
    """Total scalar parameters in the fitted pytree, less the frozen ones.

    `hmm_results.num_params` is `len(self.params)` -- a leaf count, not a
    scalar count -- so it cannot be used for the adjusted-R2 penalty.
    `n_frozen=1` is the frozen scalar `mu0`.
    """
    leaves = jax.tree_util.tree_leaves(model.params)
    return sum(int(np.size(l)) for l in leaves if hasattr(l, "shape")) - n_frozen


def suggest_covariate_design(trial: optuna.Trial):
    """Sample the covariate design: calendar terms plus a switch per weather channel.

    Returned as the keyword arguments `build_design` takes, so the whole design
    travels as one dict and `study.best_trials[...].params` already records every
    switch by name.
    """
    return dict(
        USE_OFF_DAY=trial.suggest_categorical("USE_OFF_DAY", [True, False]),
        TOD_HARMONICS=trial.suggest_int("TOD_HARMONICS", 0, 4),
        WEEK_HARMONICS=trial.suggest_int("WEEK_HARMONICS", 0, 3),
        WEATHER=tuple(
            c for c in WEATHER_COLS if trial.suggest_categorical(f"USE_{c}", [True, False])
        ),
    )


def objective(trial: optuna.Trial):
    STATES = trial.suggest_int("STATES", 2, 6)
    AR_LAGS = trial.suggest_int("AR_LAGS", 1, 6)
    design = suggest_covariate_design(trial)

    # The design first: the folds only slice it, and the seeds need the covariate count.
    ys_tv, X_raw = build_design(**design)
    trial.set_user_attr("n_covariates", int(X_raw.shape[1]))
    trial.set_user_attr("n_weather", len(design["WEATHER"]))
    trial.set_user_attr("covariates", list(design["WEATHER"]))

    # Every switch off leaves a DynamicTransition with nothing to condition on.
    if X_raw.shape[1] == 0:
        raise optuna.TrialPruned("empty covariate design")

    folds = []
    for i, (train_end, val_end) in enumerate(time_series_folds(len(ys_tv))):
        ys_fold, x_fold = fold_data(ys_tv, X_raw, train_end, val_end)
        try:
            folds.append(fold_metrics(ys_fold, x_fold, train_end, STATES, AR_LAGS))
        except optuna.TrialPruned:
            raise
        except Exception as exc:              # a diverged fit costs a trial, not the study
            trial.set_user_attr("error", f"fold {i}: {type(exc).__name__}: {exc}")
            raise optuna.TrialPruned(f"fold {i} fit failed: {type(exc).__name__}")

    return aggregate_folds(folds, trial)


def build_design(USE_OFF_DAY, TOD_HARMONICS, WEEK_HARMONICS, WEATHER=WEATHER_COLS):
    """(ys_tv, X_raw) over the pooled train+val rows -- covariates *unstandardised*.

    TOD_HARMONICS / WEEK_HARMONICS are *counts*: k means harmonics 1..k, and k = 0
    drops that block entirely. WEATHER is the subset of WEATHER_COLS to include,
    possibly empty.

    z-scoring is deliberately left to `fold_data`: each fold must standardise on its
    own training rows only, so one shared standardisation would leak later rows into
    earlier folds. Cached on the design, since NSGA-II re-samples designs and the
    weather merge_asof is the slow part of building one.
    """
    key = (bool(USE_OFF_DAY), int(TOD_HARMONICS), int(WEEK_HARMONICS), tuple(WEATHER))
    if key not in _DESIGN_CACHE:
        ys_train, ys_val, dates_tv, _ = load_split()
        X_np, _ = build_covariates(
            dates_tv,
            use_off_day=USE_OFF_DAY,
            tod_harmonics=tuple(range(1, TOD_HARMONICS + 1)),
            week_harmonics=tuple(range(1, WEEK_HARMONICS + 1)),
            weather=WEATHER,
        )
        _DESIGN_CACHE[key] = (jnp.concatenate([ys_train, ys_val]), jnp.asarray(X_np))
    return _DESIGN_CACHE[key]


def fold_data(ys_tv, X_raw, train_end, val_end):
    """This fold's series and covariates: rows [0, val_end), z-scored on [0, train_end)."""
    x_fold = standardise(X_raw[:val_end], np.arange(val_end) < train_end)
    return ys_tv[:val_end], x_fold


def fold_metrics(ys_fold, x_fold, train_end, STATES, AR_LAGS, K=VAL_K):
    """Fit on [0, train_end), score rolling K-step forecasts over the rest of the fold.

    Training and validation rows stay concatenated so a single causal forward pass
    reaches every validation anchor; filtering leaks nothing, and the covariates are
    exogenous time/weather features rather than the target. Only rows [0, train_end)
    enter the fit and the standardisation.
    """
    hmm = init_hmm(
        STATES=STATES, num_covariates=int(x_fold.shape[1]), ys_train=ys_fold[:train_end]
    )
    hmm = train_hmm(
        hmm, ys_fold[:train_end], x_fold[:train_end], STATES=STATES, AR_LAGS=AR_LAGS
    )

    y_true, y_pred, y_anchor = rolling_forecast_ar(hmm, ys_fold, x_fold, train_end, K)
    y_true, y_pred, y_anchor = map(jnp.asarray, (y_true, y_pred, y_anchor))

    return dict(
        rmse=rmse(y_true, y_pred),
        mase=mase(y_true, y_pred, y_anchor),
        mda=mda(y_true, y_pred, y_anchor),
        mase_insample=mase_insample(y_true, y_pred, ys_fold[:train_end]),
        r2=r2_score(y_true, y_pred),          # diagnostic only
        mape=mape(y_true, y_pred),
        n=int(len(y_true)),
        p=int(count_free_params(hmm)),
        log_likelihood=float(hmm.hmm_results.log_likelihood),
        converged=bool(hmm.hmm_results.convergence),
    )


def aggregate_folds(folds, trial):
    """(mean fold RMSE, mean fold MASE, mean fold MDA) -- the three objectives.

    All three average directly across folds, and that is the point of choosing them.
    RMSE is in ppm; MASE is a ratio against a baseline that faces the same window, so
    a quiet fortnight scales the numerator and denominator alike; MDA is a proportion
    of anchors. None is measured against a per-fold variance, so nothing needs a
    variance-stabilising transform before averaging -- unlike R2, whose four
    incomparable denominators forced the pooling this cell used to do.
    """
    rmses = np.array([f["rmse"] for f in folds])
    mases = np.array([f["mase"] for f in folds])
    mdas  = np.array([f["mda"] for f in folds])
    ns    = np.array([f["n"] for f in folds])
    p     = folds[0]["p"]          # same design in every fold, so the same parameter count

    trial.set_user_attr("rmse", float(rmses.mean()))
    trial.set_user_attr("rmse_std", float(rmses.std()))
    trial.set_user_attr("rmse_folds", [float(v) for v in rmses])
    trial.set_user_attr("mase", float(mases.mean()))
    trial.set_user_attr("mase_std", float(mases.std()))
    trial.set_user_attr("mase_folds", [float(v) for v in mases])
    trial.set_user_attr("mda", float(mdas.mean()))
    trial.set_user_attr("mda_std", float(mdas.std()))
    trial.set_user_attr("mda_folds", [float(v) for v in mdas])
    # Diagnostics -- recorded, not optimised.
    trial.set_user_attr("mase_insample_folds", [float(f["mase_insample"]) for f in folds])
    trial.set_user_attr("r2_folds", [float(f["r2"]) for f in folds])
    trial.set_user_attr("mape", float(np.mean([f["mape"] for f in folds])))
    trial.set_user_attr("log_likelihood", [f["log_likelihood"] for f in folds])
    trial.set_user_attr("converged", all(f["converged"] for f in folds))
    trial.set_user_attr("n_params", int(p))
    trial.set_user_attr("n_val_anchors", [int(v) for v in ns])

    if not (np.isfinite(rmses).all() and np.isfinite(mases).all() and np.isfinite(mdas).all()):
        raise optuna.TrialPruned("non-finite validation metric")
    return float(rmses.mean()), float(mases.mean()), float(mdas.mean())


def init_hmm(STATES, num_covariates, ys_train):
    """The Gaussian + covariate HMM, seeded exactly as in the week-5 fit cell."""
    mu_seed = jnp.quantile(ys_train, jnp.linspace(0.05, 0.95, STATES)).at[0].set(450)
    sigma_seed = jnp.std(ys_train) * jnp.ones(STATES)
    tm_seed = jnp.full((STATES, STATES), 0.1).at[jnp.diag_indices(STATES)].set(0.7)
    beta_init = jnp.zeros((num_covariates, STATES, STATES - 1))

    return HMM(
        emission=GaussEmission.from_params(mu_seed, sigma_seed),
        transition=DynamicTransition(transition_matrix_to_logits(tm_seed), beta_init),
        # A DynamicTransition has no time-invariant matrix, so no stationary distribution.
        inital_distribution=jnp.full(STATES, 1.0 / STATES),
    )


def train_hmm(hmm, ys_train, x_train_std, STATES, AR_LAGS):
    """Fit the Gaussian model, warm-start the AR model from it, return the AR fit.

    At phi = 0 the AR model *is* the fitted Gaussian one, so the second fit can
    only improve on the first.
    """
    hmm.fit(ys=ys_train, ts=None, xs=x_train_std, tol=FIT_TOL, frozen={"mu0": False})

    ar_hmm = HMM(
        emission=AutoregressiveGaussEmission.from_params(
            state_means(hmm.emission, ys_train),
            jnp.exp(hmm.emission.log_sigma),
            jnp.zeros((AR_LAGS, STATES)),
        ),
        transition=DynamicTransition(hmm.transition.transition_logits, hmm.transition.beta),
        inital_distribution=jnp.full(STATES, 1.0 / STATES),
    )
    ar_hmm.fit(ys=ys_train, ts=None, xs=x_train_std, tol=FIT_TOL, frozen={"mu0": False})
    return ar_hmm


## Optuna study

Three objectives, all averaged over the 4 expanding-window folds: minimise 6 h-ahead **RMSE**,
minimise **MASE**, maximise **MDA**. They measure different failures — magnitude in ppm, skill
against a random walk, and directional agreement — so the Pareto front is a genuine trade-off.
Expect it to be wider than a two-objective front: with three criteria more designs are
non-dominated.

Note that none of the three penalises parameter count, which adjusted R2 previously did. `n_params`
is still recorded on every trial, so read it off the front when choosing — a design that wins by a
hair with three times the parameters is not the one to refit. That also means no
`study.best_params` and no report-based pruning: read `study.best_trials`.

The study name is new (`week5_room009_ar_cov_cv4m`): it has three objectives where the earlier
studies had two, so their trials cannot be loaded into it at all. `week5_room009_ar_cov_cv4` (fold-
mean R2) and `week5_room009_ar_cov` (single split) stay in the same file for reference. Both live in `results/optuna/week5_room009.db`, so the
old trials stay readable. A second `optimize` call **extends** the CV study rather than restarting
it; delete the db, or use a fresh study name, for a clean run.


In [8]:
import os

os.makedirs("results/optuna", exist_ok=True)

study = optuna.create_study(
    directions=["minimize", "minimize", "maximize"],   # RMSE down, MASE down, MDA up
    sampler=optuna.samplers.NSGAIISampler(seed=0, population_size=16),
    study_name="week5_room009_ar_cov_cv4m",        # CV-scored; the old single-split study is separate
    storage="sqlite:///results/optuna/week5_room009.db",
    load_if_exists=True,
)
study.optimize(objective, n_trials=N_TRIALS, gc_after_trial=True)

trials_df = study.trials_dataframe()
trials_df


[I 2026-09-25 10:29:43,188] A new study created in RDB with name: week5_room009_ar_cov_cv4m
[I 2026-09-25 10:30:05,334] Trial 0 pruned. non-finite validation metric
[I 2026-09-25 10:30:28,610] Trial 1 finished with values: [413.63183892914594, 1.6674520493463558, 0.6780360043048859] and parameters: {'STATES': 5, 'AR_LAGS': 6, 'USE_OFF_DAY': True, 'TOD_HARMONICS': 2, 'WEEK_HARMONICS': 3, 'USE_mean_temp': False, 'USE_mean_relative_hum': False, 'USE_mean_wind_speed': True, 'USE_mean_pressure': False, 'USE_mean_cloud_cover': False, 'USE_mean_radiation': False}.
[I 2026-09-25 10:30:48,417] Trial 2 finished with values: [400.53831471133714, 1.6322317398791413, 0.6892803758382797] and parameters: {'STATES': 5, 'AR_LAGS': 4, 'USE_OFF_DAY': True, 'TOD_HARMONICS': 1, 'WEEK_HARMONICS': 1, 'USE_mean_temp': True, 'USE_mean_relative_hum': False, 'USE_mean_wind_speed': True, 'USE_mean_pressure': False, 'USE_mean_cloud_cover': True, 'USE_mean_radiation': True}.
[I 2026-09-25 10:31:06,301] Trial 3 fini

,number,values_0,values_1,values_2,datetime_start,datetime_complete,duration,params_AR_LAGS,params_STATES,params_TOD_HARMONICS,...,user_attrs_n_covariates,user_attrs_n_params,user_attrs_n_val_anchors,user_attrs_n_weather,user_attrs_r2_folds,user_attrs_rmse,user_attrs_rmse_folds,user_attrs_rmse_std,system_attrs_NSGAIISampler:generation,state
0,0,NaN,NaN,NaN,2026-09-25 10:29:43.191956,2026-09-25 10:30:05.333458,0 days 00:00:22.141502,5,4,2,...,11,171,"[667, 667, 667, 667]",2,"[nan, -1.0135088782822357, -1.5467002935097054...",NaN,"[nan, 484.14900078996016, 310.62877773145874, ...",NaN,0,PRUNED
1,1,413.631839,1.667452,0.678036,2026-09-25 10:30:05.399873,2026-09-25 10:30:28.609056,0 days 00:00:23.209183,6,5,2,...,12,299,"[667, 667, 667, 667]",1,"[-0.6674747688818989, -1.8074319536861756, 0.0...",413.631839,"[568.861403580362, 571.6851151864271, 185.0099...",164.706177,0,COMPLETE
2,2,400.538315,1.632232,0.689280,2026-09-25 10:30:28.693656,2026-09-25 10:30:48.416550,0 days 00:00:19.722894,4,5,1,...,9,229,"[667, 667, 667, 667]",4,"[-0.0045667367268587, -1.1464176121642793, -1....",400.538315,"[441.53611268228184, 499.87265183040574, 287.8...",79.062582,0,COMPLETE
3,3,400.375297,1.679347,0.685157,2026-09-25 10:30:48.513533,2026-09-25 10:31:06.300432,0 days 00:00:17.786899,1,3,2,...,10,74,"[667, 667, 667, 667]",5,"[-0.8060314770940273, -0.5942958246593946, -0....",400.375297,"[592.0242422116345, 430.81093657646096, 258.63...",126.689353,0,COMPLETE
4,4,409.992909,1.922603,0.670915,2026-09-25 10:31:06.418549,2026-09-25 10:31:31.613099,0 days 00:00:25.194550,4,6,1,...,6,245,"[667, 667, 667, 667]",3,"[0.23046520921466973, -1.4890883792723217, -3....",409.992909,"[386.44790965902763, 538.2975758550089, 393.22...",79.119365,0,COMPLETE
5,5,386.589941,1.876223,0.649175,2026-09-25 10:31:31.790581,2026-09-25 10:31:46.052772,0 days 00:00:14.262191,5,3,1,...,6,62,"[667, 667, 667, 667]",4,"[0.19172519268008925, -0.9691923237090074, -0....",386.589941,"[396.05577868862304, 478.7913905578846, 258.73...",80.035581,0,COMPLETE
6,6,426.612434,1.989765,0.660795,2026-09-25 10:31:46.224962,2026-09-25 10:32:01.278781,0 days 00:00:15.053819,6,3,3,...,13,107,"[667, 667, 667, 667]",5,"[-0.2503391788496263, -1.6469973981181814, -0....",426.612434,"[492.59585470195634, 555.1099637577199, 251.73...",113.832663,0,COMPLETE
7,7,NaN,NaN,NaN,2026-09-25 10:32:01.423049,2026-09-25 10:38:29.637395,0 days 00:06:28.214346,2,2,3,...,11,31,"[667, 667, 667, 667]",2,"[nan, -0.07184731810609457, -1.251695529818179...",NaN,"[nan, 353.23906637190316, 292.0838883166532, 3...",NaN,0,PRUNED
8,8,440.212732,1.901383,0.668666,2026-09-25 10:38:29.838834,2026-09-25 10:38:56.548765,0 days 00:00:26.709931,5,6,3,...,15,521,"[667, 667, 667, 667]",3,"[-0.8919234749922973, -0.744644911529512, -2.5...",440.212732,"[605.9385954605667, 450.667032547022, 365.5416...",104.228375,0,COMPLETE
9,9,362.594382,1.460628,0.676162,2026-09-25 10:38:56.761899,2026-09-25 10:39:21.764928,0 days 00:00:25.003029,6,5,0,...,8,219,"[667, 667, 667, 667]",5,"[-0.46914503727835233, 0.3868439368248703, -1....",362.594382,"[533.9605468173733, 267.1700544585869, 319.770...",101.738438,0,COMPLETE


In [9]:
print(f"{len(study.trials)} trials, {len(study.best_trials)} on the Pareto front")
print(f"every metric is a mean over the {N_FOLDS} expanding-window folds; +- is the spread\n")
print("  MASE < 1 beats a 6 h-ahead random walk; MDA > 0.5 beats a coin flip\n")

def _design_str(t):
    """The sampled design as one line: calendar switches, then the chosen weather."""
    p = t.params
    calendar = f"off_day={p['USE_OFF_DAY']}, tod={p['TOD_HARMONICS']}, week={p['WEEK_HARMONICS']}"
    weather = ", ".join(t.user_attrs.get("covariates", [])) or "none"
    return f"S={p['STATES']}, lags={p['AR_LAGS']}, {calendar} | weather: {weather}"


front = sorted(study.best_trials, key=lambda t: t.values[0])
for t in front:
    a = t.user_attrs
    rmse_f = " ".join(f"{v:.0f}" for v in a["rmse_folds"])
    mase_f = " ".join(f"{v:.2f}" for v in a["mase_folds"])
    mda_f  = " ".join(f"{v:.2f}" for v in a["mda_folds"])
    print(f"trial {t.number:>3}  RMSE {t.values[0]:7.1f} +-{a['rmse_std']:5.1f}   "
          f"MASE {t.values[1]:5.2f} +-{a['mase_std']:4.2f}   "
          f"MDA {t.values[2]:5.3f} +-{a['mda_std']:4.3f}   "
          f"(p={a['n_params']:>4}, D={a['n_covariates']:>2})\n"
          f"            {_design_str(t)}\n"
          f"            per fold -- RMSE: {rmse_f} | MASE: {mase_f} | MDA: {mda_f}")

beats_naive = [t for t in front if t.values[1] < 1]
print(f"\n{len(beats_naive)}/{len(front)} front members beat the naive random walk (MASE < 1)")

best = front[0]
print(f"Lowest-RMSE front member: trial {best.number} -> {_design_str(best)}")
print("  NOTE: this is one corner of a three-objective front, not 'the' optimum -- other")
print("  members trade ppm accuracy for directional accuracy or for fewer parameters.")
print("  Prefer a member whose per-fold values are tight over one that wins on the mean alone.")


60 trials, 5 on the Pareto front
every metric is a mean over the 4 expanding-window folds; +- is the spread

  MASE < 1 beats a 6 h-ahead random walk; MDA > 0.5 beats a coin flip

trial  46  RMSE   284.7 +-108.1   MASE  1.22 +-0.54   MDA 0.606 +-0.071   (p=  95, D= 1)
            S=6, lags=4, off_day=False, tod=0, week=0 | weather: mean_radiation
            per fold -- RMSE: 321 215 160 443 | MASE: 0.77 1.31 2.07 0.72 | MDA: 0.70 0.54 0.54 0.65
trial  29  RMSE   340.4 +- 88.9   MASE  1.34 +-0.56   MDA 0.683 +-0.086   (p= 127, D= 7)
            S=4, lags=6, off_day=True, tod=0, week=1 | weather: mean_temp, mean_relative_hum, mean_wind_speed, mean_radiation
            per fold -- RMSE: 468 292 231 372 | MASE: 1.02 1.67 2.05 0.61 | MDA: 0.78 0.59 0.60 0.76
trial  16  RMSE   341.4 +- 83.8   MASE  1.50 +-0.89   MDA 0.717 +-0.096   (p= 203, D=14)
            S=4, lags=4, off_day=True, tod=4, week=0 | weather: mean_temp, mean_wind_speed, mean_pressure, mean_cloud_cover, mean_radiation
     

## Notes

- Each harmonic adds 2 covariates and therefore `2 * STATES * (STATES - 1)` `beta` parameters, so
  `TOD_HARMONICS` and `STATES` interact strongly in the penalty. At `STATES = 6` with the full
  design, `beta` alone approaches the per-fold validation anchor count and adjusted R2 goes
  negative — the penalty judging that corner of the space unusable rather than merely worse.
- Each included weather channel adds one covariate and so `STATES * (STATES - 1)` `beta`
  parameters — the same penalty a harmonic pays per column. With the six channels switched
  independently the design space is `2^6` weather subsets times the calendar options, which is why
  the search is run with NSGA-II rather than a grid.
- Weather inclusion is searched, not assumed: a front member that drops most of the block is saying
  those channels did not pay for their parameters at that state count, not that weather is
  irrelevant in general.
- All three reported metrics are **means over the 4 folds**; `rmse_folds`, `mase_folds` and
  `mda_folds` hold the individual values. A design whose folds disagree is a worse bet than its
  mean suggests, so read the spread before picking.
- Absolute R2 levels are not comparable with the single-split numbers of the earlier study, which
  scored only the fold-4 window -- the most variable, and so the most flattering, of the four.
- MASE is scaled by the **horizon-matched** random walk: the baseline forecast for anchor t is
  y_t itself, carried 6 hours forward, scored on the same anchors. That is what makes "< 1 beats
  naive" true as stated. The Hyndman-Koehler version, scaled by the in-sample one-step naive MAE,
  is recorded separately as `mase_insample_folds`; it will read far above 1 because its denominator
  is a one-step error against a twelve-step numerator, and it is comparable with the literature
  rather than with the baseline that matters here.
- Beating the random walk at 6 hours is a real bar: CO2 is strongly autocorrelated, so y_t is a
  good predictor of y_{t+12} whenever the room's occupancy does not change in between. A MASE
  slightly above 1 means the model is adding structure that does not pay for itself at this
  horizon; the place it should win is precisely the transitions the random walk cannot see coming.
- `r2_folds` still records the per-fold determination R2 as a diagnostic. Expect it to be negative
  on folds 1-3 even for good designs: those validation blocks have standard deviations of roughly
  441 / 341 / 195 ppm against fold 4's 568, and the fold-3 window is quiet enough that the
  lowest-RMSE forecast in the study still loses to its flat mean. That is a property of the
  windows, not of the model, and it is exactly why MASE replaced it as the objective.
- Fold 1 trains on only ~1/5 of the pooled series (~26 days at 30 min). A weekly harmonic is
  estimable there, but barely — a design that is fine in folds 2–4 and poor in fold 1 is more
  likely short of training data than genuinely bad.
- The folds partition train+val only. The test split is still untouched by anything in this
  notebook.
- The front is a trade-off, not a ranking: its low-RMSE end buys a few percent of accuracy with
  three times the parameters. Pick the member that suits what the result is for, and refit it in
  `week_5.ipynb` to get the diagnostics and the (still untouched) test-set numbers.
